# Exploratory analysis of IBM HI-Small

<div class="eda-draft">Draft for Brand, Legal, Risk and Independence review</div>

Presentation follows the user-selected [Deloitte Data Design system](https://claude.ai/artifact/8LsWY495eAgpPAUWCxd336). This independent research notebook does not claim Deloitte authorship or endorsement.

**Full ledger, derived account roles, and the saved candidate scan.** This notebook explains the data and examines why the current business → business → person rule selected 905 chains but none of the 5,177 laundering-labeled transactions.

The reference label identifies synthetic money laundering. It does not identify confirmed embezzlement, employee ownership, a shell company, or a false invoice. Labels are used here for retrospective exploration, never to select or rank the saved candidates.

Three data-science reviews extend the descriptive profile with distribution-shift analysis, account/entity network measures, and a chronological anomaly-ranking experiment. Each section states its population, units, assumptions, and limits. The anomaly experiment fits on earlier dates and evaluates later dates; it does not change the saved path-selection rule.

Run cells from top to bottom with this repository's `.venv` Python kernel. Install the optional analysis dependencies with `python -m pip install -e '.[eda]'`. The full dataset and prior scan outputs must already exist; this notebook does not download, import, or rerun the transaction-path scan. See `notebooks/README.md` for setup and input paths.

In [ ]:
import os
import sys
import json
import sqlite3
import hashlib
import platform
from pathlib import Path
from importlib.metadata import version

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

def find_repository():
    override = os.environ.get('EMBEZZLEMENT_REPO_ROOT')
    candidates = [Path(override).expanduser()] if override else [Path.cwd(), *Path.cwd().parents]
    for candidate in candidates:
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'embezzlement_analysis').is_dir():
            return candidate.resolve()
    raise FileNotFoundError('Open this notebook inside the repository, or set EMBEZZLEMENT_REPO_ROOT.')

ROOT = find_repository()
DB_PATH = ROOT / 'data/hi-small.sqlite'
REPORT_DIR = ROOT / 'reports'
RAW_DIR = ROOT / 'data/raw/full-hi-small'
FIGURE_DIR = REPORT_DIR / 'eda/figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

required = [DB_PATH, REPORT_DIR / 'hi-small-audit.json', REPORT_DIR / 'hi-small-scan.json',
            REPORT_DIR / 'hi-small-candidates.jsonl', REPORT_DIR / 'hi-small-run-provenance.json',
            RAW_DIR / 'acquisition.json']
for source_name in ['HI-Small_Trans.csv', 'HI-Small_accounts.csv', 'HI-Small_Patterns.txt']:
    required.extend([RAW_DIR / source_name, RAW_DIR / (source_name + '.source.json')])
missing = [str(path.relative_to(ROOT)) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing full-run inputs: ' + ', '.join(missing) + '. See notebooks/README.md.')

AUDIT = json.loads((REPORT_DIR / 'hi-small-audit.json').read_text())
SCAN = json.loads((REPORT_DIR / 'hi-small-scan.json').read_text())
ACQUISITION = json.loads((RAW_DIR / 'acquisition.json').read_text())
verification_path = REPORT_DIR / 'hi-small-final-verification.json'
VERIFY = json.loads(verification_path.read_text()) if verification_path.exists() else {}
conn = sqlite3.connect(DB_PATH.resolve().as_uri() + '?mode=ro', uri=True)
conn.execute('PRAGMA query_only=ON')
# One consistent read snapshot across all EDA queries. No database writes occur.
conn.execute('BEGIN')
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', 80)

sys.path.insert(0, str(ROOT / 'notebooks'))
from eda_design import (TOKENS, GROUPS, FOCUS, REST, TEXT, TEXT_MUTED, LINE,
                        show_table, install_notebook_style, finish_figure as render_figure)
install_notebook_style()
FIGURES = []
def finish_figure(fig, name, title, subtitle='', note='', height=500):
    FIGURES.append(render_figure(fig, name, title, subtitle, note, height, output_dir=FIGURE_DIR))

environment = pd.DataFrame([
    {'Component': 'Python', 'Version': platform.python_version()},
    *[{'Component': package, 'Version': version(package)} for package in
      ['pandas', 'numpy', 'plotly', 'scipy', 'scikit-learn', 'nbformat', 'nbclient', 'ipykernel']],
])
show_table(environment)
display(Markdown('Database access is read-only. Full-ledger counts use SQL aggregation; SQL aggregates, scoped account-day profiles, a bounded USD reference and saved candidates enter pandas. Source amounts remain text/Decimal; plotting conversions are labeled.'))

## Source integrity and analysis scope

This notebook examines the complete **HI-Small v8** transaction snapshot, account metadata and published laundering-attempt annotations. The saved scan applies a predefined two-payment **business → business → person** rule. It produces review leads, not findings of embezzlement.

The next cell recomputes ledger row/label counts from SQLite and checks the audit, scan and acquisition manifests. It hashes the small audit/scan reports. **Raw-file SHA-256 values are inherited from completed acquisition and verification; this notebook does not rehash the 475 MB transaction CSV or account CSV by default.** Agreement among manifests confirms consistency of saved records, not that raw bytes could never have changed.

In [ ]:
import hashlib
import json
context_raw_dir = ROOT / 'data' / 'raw' / 'full-hi-small'
context_acquisition = json.loads((context_raw_dir / 'acquisition.json').read_text())
context_run = json.loads((REPORT_DIR / 'hi-small-run-provenance.json').read_text())
assert context_acquisition['status'] == 'complete_verified'
context_acquisition_by_name = {r['file_name']: r for r in context_acquisition['files']}
for context_name in ['hi-small-audit.json', 'hi-small-scan.json']:
    assert hashlib.sha256((REPORT_DIR / context_name).read_bytes()).hexdigest() == context_run['artifacts'][context_name]['sha256']
context_db_counts = pd.read_sql_query("""
    SELECT COUNT(*) AS rows, COALESCE(SUM(is_laundering=0),0) AS label_0,
           COALESCE(SUM(is_laundering=1),0) AS label_1 FROM transactions
""", conn).iloc[0]
assert int(context_db_counts['rows']) == AUDIT['transactions']['rows'] == SCAN['coverage']['transaction_count']
for context_label in ['0', '1']:
    assert int(context_db_counts['label_'+context_label]) == AUDIT['transactions']['label_counts'][context_label]
assert int(context_db_counts['label_0'] + context_db_counts['label_1']) == int(context_db_counts['rows'])
assert int(context_db_counts['label_1']) == SCAN['ledger_label_diagnostics']['laundering']
context_db_sources = pd.read_sql_query('SELECT file_name,sha256,row_count FROM sources ORDER BY sha256',conn)
assert len(context_db_sources) == 1, 'Expected the separate full HI-Small database.'
assert context_db_sources.iloc[0]['sha256'] == AUDIT['sources']['transactions']['sha256'] == SCAN['sources']['transactions'][0]['sha256']
assert int(context_db_sources.iloc[0]['row_count']) == int(context_db_counts['rows'])
assert SCAN['sources']['accounts']['sha256'] == AUDIT['sources']['accounts']['sha256']
context_record_counts = {'transactions':AUDIT['transactions']['rows'], 'accounts':AUDIT['accounts']['rows'],
                         'patterns':AUDIT['patterns']['transaction_rows']}
context_source_rows = []
for context_kind, context_source in AUDIT['sources'].items():
    context_manifest = context_acquisition_by_name[context_source['file_name']]
    context_sidecar = json.loads((context_raw_dir / (context_source['file_name']+'.source.json')).read_text())
    assert context_manifest['sha256'] == context_sidecar['sha256'] == context_source['sha256']
    assert context_manifest['status'] == context_sidecar['status'] == 'complete_verified'
    assert context_manifest['size_bytes'] == context_sidecar['size_bytes'] == context_source['bytes']
    assert (context_raw_dir / context_source['file_name']).stat().st_size == context_source['bytes']
    context_source_rows.append({'Source':context_source['file_name'], 'Records':context_record_counts[context_kind],
                                'Bytes':context_source['bytes'], 'SHA-256 (previously verified)':context_source['sha256']})
show_table(pd.DataFrame(context_source_rows), formats={'Records':'{:,.0f}', 'Bytes':'{:,.0f}'})
display(Markdown(f"**Fresh SQLite check passed:** {int(context_db_counts['rows']):,} rows; "
                 f"{int(context_db_counts['label_1']):,} labelled laundering; "
                 f"{int(context_db_counts['label_0']):,} labelled non-laundering. "
                 "Pattern records exclude BEGIN/END markers. Raw hashes were not recomputed here."))

## Account roles and source coverage

Synthetic name prefixes determine derived roles: `Corporation`, `Partnership` and `Sole Proprietorship` become **business**, `Individual` becomes **person**, and other names remain **unknown**. These account classifications do not establish legal form, employment, or shell status. Multiple accounts can belong to one entity.

Counts below reuse the independent account-file audit. Complete endpoint-to-metadata matching does **not** mean that every endpoint has a known role. The chart uses a logarithmic count axis to expose the small person/unknown groups; use printed counts and shares rather than interpreting bar lengths as linear proportions.

In [ ]:
context_role_counts = AUDIT['accounts']['row_role_counts_derived_from_name']
context_roles = pd.DataFrame([{'Derived role':r, 'Accounts':context_role_counts.get(r,0),
                              'Share of account rows':context_role_counts.get(r,0)/AUDIT['accounts']['rows']}
                             for r in ['business','unknown','person']])
assert int(context_roles['Accounts'].sum()) == AUDIT['accounts']['rows']
assert context_role_counts == SCAN['coverage']['account_roles']
show_table(context_roles, formats={'Accounts':'{:,.0f}', 'Share of account rows':'{:.3%}'})
context_fig = go.Figure(go.Bar(
    x=context_roles['Accounts'],y=context_roles['Derived role'],orientation='h',
    marker=dict(color=[FOCUS if role=='person' else REST for role in context_roles['Derived role']],
                line=dict(width=0)),
    text=context_roles['Accounts'].map(lambda count:f'{count:,}'),textposition='outside',
    textfont=dict(color=TEXT),cliponaxis=False,
    customdata=context_roles['Share of account rows'],
    hovertemplate=('<b>%{y}</b><br>Account rows: %{x:,}'
                   '<br>Share of account rows: %{customdata:.3%}<extra></extra>'),
    showlegend=False))
context_fig.update_xaxes(title_text='Account rows (count)',
    range=[0,float(context_roles['Accounts'].max())*1.18],showgrid=False)
context_fig.update_yaxes(autorange='reversed',showgrid=False,
    categoryorder='array',categoryarray=context_roles['Derived role'].tolist())
finish_figure(context_fig,'account-role-coverage',
    title='Very few account rows are classified as person',
    subtitle=f"All {AUDIT['accounts']['rows']:,} metadata account rows · derived roles",
    note='Roles come from simulator entity names. Unknown roles stay unknown; role classification is not verified ownership.',
    height=360)
context_n = AUDIT['transactions']['rows']
context_quality = pd.DataFrame([
    {'Measure':'Endpoint occurrences matched to unambiguous metadata',
     'Count':AUDIT['transactions']['endpoint_occurrence_coverage']['unambiguous_metadata'],
     'Denominator':2*context_n, 'Meaning':'Sender plus recipient occurrences; accounts repeat'},
    {'Measure':'Endpoint occurrences with unknown derived role',
     'Count':AUDIT['transactions']['endpoint_occurrence_roles_derived_from_name']['unknown'],
     'Denominator':2*context_n, 'Meaning':'Metadata exists; role-based selection excludes these endpoints'},
    {'Measure':'Self-transfers by exact bank/account identity',
     'Count':AUDIT['transactions']['self_transfer_rows'], 'Denominator':context_n,
     'Meaning':'Observable structure, not an independent fraud indicator'},
    {'Measure':'Adjacent timestamp reversals in original CSV order',
     'Count':AUDIT['transactions']['adjacent_timestamp_reversals'], 'Denominator':context_n-1,
     'Meaning':'File order is not chronology'}])
context_quality['Share'] = context_quality['Count']/context_quality['Denominator']
show_table(context_quality, formats={'Count':'{:,.0f}','Denominator':'{:,.0f}','Share':'{:.2%}'})
display(Markdown('The audit reports **740 person accounts**, **6,759 unknown-role accounts**, and '
                 '**2,418,736 adjacent timestamp reversals**. The detector orders connected transfers '
                 'by timestamps, not file position. Neither reversals nor self-transfers establish misconduct.'))

## Publisher main window versus the observed tail

The saved publisher description distinguishes the main period ending **2022-09-10** from later transactions and says the later records are all laundering. We retain that claim as source context and compare it with actual v8 labels. The following table aggregates the independent CSV audit's daily counts; the daily-profile section separately queries SQLite. These retrospective groups are not a train/test split.

In [ ]:
context_cutoff_date = AUDIT['transactions']['publisher_main_period_end']
context_days = pd.DataFrame(AUDIT['transactions']['daily_labels'])
context_days['Period'] = np.where(context_days['date']<=context_cutoff_date,
                                 'Main window: Sep 1–10','Tail: Sep 11–18')
context_periods = context_days.groupby('Period',sort=False)[['rows','label_0','label_1']].sum().reset_index()
context_periods['Laundering-label share'] = context_periods['label_1']/context_periods['rows']
assert int(context_periods['rows'].sum()) == int(context_db_counts['rows'])
assert int(context_periods['label_1'].sum()) == int(context_db_counts['label_1'])
context_tail = context_periods.loc[context_periods['Period'].str.startswith('Tail')].iloc[0]
assert int(context_tail['rows']) == AUDIT['transactions']['after_publisher_main_window']['rows']
show_table(context_periods.rename(columns={'rows':'Transactions','label_0':'Label 0','label_1':'Label 1'}),
           formats={'Transactions':'{:,.0f}','Label 0':'{:,.0f}','Label 1':'{:,.0f}','Laundering-label share':'{:.3%}'})
display(Markdown(f"**Observed tail: {int(context_tail['label_1']):,} / {int(context_tail['rows']):,} "
                 f"rows carry label 1 ({context_tail['Laundering-label share']:.2%}), not 100%.** "
                 f"The remaining {int(context_tail['label_0']):,} rows carry label 0. "
                 "This sharp change in prevalence makes one pooled rate insufficient to describe the dataset. "
                 "No calendar rule overwrites or infers labels."))

## Time, categories and amounts in the full ledger

These cells recompute summaries from the read-only database, restricted to the transaction source identified by the independent audit. They load grouped results, not the full 5.08-million-row ledger. `Is Laundering` is a synthetic AML annotation, **not an embezzlement outcome**. Labels support retrospective exploration here; they do not select detection leads.

In [ ]:
SOURCE_SHA = AUDIT["sources"]["transactions"]["sha256"]
MAIN_CUTOFF = pd.Timestamp("2022-09-11")
daily = pd.read_sql_query(
    """SELECT substr(timestamp, 1, 10) AS date, COUNT(*) AS rows,
              SUM(is_laundering) AS label_1,
              SUM(CASE WHEN is_laundering = 0 THEN 1 ELSE 0 END) AS label_0
       FROM transactions WHERE source_sha256 = ?
       GROUP BY substr(timestamp, 1, 10) ORDER BY date""",
    conn, params=(SOURCE_SHA,),
)
audit_daily = pd.DataFrame(AUDIT["transactions"]["daily_labels"])
compare_columns = ["date", "rows", "label_0", "label_1"]
pd.testing.assert_frame_equal(
    daily[compare_columns].reset_index(drop=True),
    audit_daily[compare_columns].reset_index(drop=True), check_dtype=False,
)
assert int(daily["rows"].sum()) == AUDIT["transactions"]["rows"]
daily["date"] = pd.to_datetime(daily["date"])
daily["label_rate_pct"] = 100 * daily["label_1"] / daily["rows"]
daily["phase"] = np.where(daily["date"] < MAIN_CUTOFF, "Main window", "Tail")
phase_summary = daily.groupby("phase", sort=False)[["rows", "label_0", "label_1"]].sum()
phase_summary["Label rate (%)"] = 100 * phase_summary["label_1"] / phase_summary["rows"]
show_table(phase_summary.rename(columns={"rows":"Transactions", "label_0":"Label 0", "label_1":"Label 1"}))
display(Markdown("Daily SQL counts and labels match the independent CSV audit exactly. The September 11 boundary follows the publisher's stated September 1–10 main window; it is not fitted to the labels."))

In [ ]:
tick_positions = np.unique(np.linspace(0, len(daily) - 1, min(7, len(daily)), dtype=int))
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.17,
                    subplot_titles=("Transaction volume (log scale)", "Laundering label rate (%)"))
fig.add_trace(go.Bar(
    x=daily["date"], y=daily["rows"], marker_color=REST, marker_line_width=0,
    width=0.5 * 86_400_000, name="Transactions", showlegend=False,
    customdata=daily[["label_0", "label_1"]].to_numpy(),
    hovertemplate="%{x|%d %b %Y}<br>Transactions: %{y:,}<br>Label 0: %{customdata[0]:,}<br>Label 1: %{customdata[1]:,}<extra></extra>",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=daily["date"], y=daily["label_rate_pct"], mode="lines",
    line=dict(color=FOCUS, width=2), name="Label rate", showlegend=False,
    customdata=daily[["label_1", "rows"]].to_numpy(),
    hovertemplate="%{x|%d %b %Y}<br>Label rate: %{y:.4f}%<br>Label 1: %{customdata[0]:,}<br>Transactions: %{customdata[1]:,}<extra></extra>",
), row=2, col=1)
fig.update_yaxes(type="log", title_text="Transactions", row=1, col=1)
fig.update_yaxes(title_text="Label rate (%)", ticksuffix="%", rangemode="tozero", row=2, col=1)
fig.update_xaxes(
    tickmode="array", tickvals=daily["date"].iloc[tick_positions], tickformat="%d %b", tickangle=0,
    range=[daily["date"].iloc[0] - pd.Timedelta(days=0.6), daily["date"].iloc[-1] + pd.Timedelta(days=0.6)],
)
fig.update_xaxes(title_text="Transaction date in source clock; no timezone inferred", row=2, col=1)
fig.add_annotation(
    x=daily["date"].iloc[-1], y=daily["label_rate_pct"].iloc[-1],
    text=f"{daily['label_rate_pct'].iloc[-1]:.1f}%", showarrow=False,
    xanchor="right", yshift=16, row=2, col=1,
)
finish_figure(
    fig, "eda-daily-volume-label-rate",
    title="Volume falls after 10 September while the positive-label share rises",
    subtitle="Full ledger · daily transaction counts (log scale) and laundering label rate (%)",
    note="Main window: 1–10 September; tail: 11–18 September. Panels use different units. Counts include all source dates.",
    height=650,
)
display(Markdown("The panels use different units: transaction counts on a logarithmic scale above, label percentage on a linear scale below. The tail contains both labels; do not treat its date as a perfect laundering indicator."))

## Currency and payment format

One SQL aggregation counts each `(paid currency, payment format, label)` group. The tables and plots below derive their margins from that result. **Counts do not add monetary values across currencies.** Category label rates describe this synthetic ledger; they are not causal effects or calibrated fraud risks.

In [ ]:
category_joint = pd.read_sql_query(
    """SELECT payment_currency, payment_format, is_laundering, COUNT(*) AS transactions
       FROM transactions WHERE source_sha256 = ?
       GROUP BY payment_currency, payment_format, is_laundering""",
    conn, params=(SOURCE_SHA,),
)
assert int(category_joint["transactions"].sum()) == int(daily["rows"].sum())
category_joint["positive_transactions"] = category_joint["transactions"] * category_joint["is_laundering"]
def category_margin(column):
    out = category_joint.groupby(column, as_index=False)[["transactions", "positive_transactions"]].sum()
    out["label_rate_pct"] = 100 * out["positive_transactions"] / out["transactions"]
    return out.sort_values(["transactions", column], ascending=[False, True]).reset_index(drop=True)
currency_profile = category_margin("payment_currency")
format_profile = category_margin("payment_format")
assert dict(zip(currency_profile["payment_currency"], currency_profile["transactions"])) == AUDIT["transactions"]["payment_currency_counts"]
assert dict(zip(format_profile["payment_format"], format_profile["transactions"])) == AUDIT["transactions"]["payment_format_counts"]
assert int(category_joint["positive_transactions"].sum()) == int(daily["label_1"].sum())

In [ ]:
display(Markdown("**Eight largest paid-currency groups by transaction count** (remaining currencies stay in `currency_profile`)."))
show_table(currency_profile.head(8).rename(columns={"payment_currency":"Paid currency", "transactions":"Transactions", "positive_transactions":"Label 1", "label_rate_pct":"Label rate (%)"}))
display(Markdown("**All payment formats**, ordered by transaction count."))
show_table(format_profile.rename(columns={"payment_format":"Payment format", "transactions":"Transactions", "positive_transactions":"Label 1", "label_rate_pct":"Label rate (%)"}))

In [ ]:
fig = make_subplots(rows=2, cols=2, horizontal_spacing=0.28, vertical_spacing=0.22,
                    subplot_titles=("Eight largest paid currencies: volume", "Eight largest paid currencies: label share",
                                    "All payment formats: volume", "All payment formats: label share"))
for row, frame, key in [
    (1, currency_profile.head(8), "payment_currency"),
    (2, format_profile, "payment_format"),
]:
    names = frame[key].tolist()
    for col, measure, colour in [(1, "transactions", REST), (2, "label_rate_pct", FOCUS)]:
        fig.add_trace(go.Bar(
            x=frame[measure], y=names, orientation="h", marker_color=colour,
            marker_line_width=0, showlegend=False,
            customdata=frame[["transactions", "positive_transactions", "label_rate_pct"]].to_numpy(),
            hovertemplate="%{y}<br>Transactions: %{customdata[0]:,}<br>Label 1: %{customdata[1]:,}<br>Label rate: %{customdata[2]:.4f}%<extra></extra>",
        ), row=row, col=col)
        fig.update_yaxes(autorange="reversed", categoryorder="array", categoryarray=names, showgrid=False, row=row, col=col)
        fig.update_xaxes(rangemode="tozero", showgrid=False,
                         title_text="Transactions" if col == 1 else "Laundering label rate (%)",
                         tickformat="~s" if col == 1 else ".2f", ticksuffix="" if col == 1 else "%",
                         row=row, col=col)
fig.update_layout(bargap=0.5)
finish_figure(
    fig, "eda-category-volume-label-rate",
    title="Transaction volume and positive-label share tell different category stories",
    subtitle="Full ledger · eight largest paid-currency groups and all payment formats · counts and label rate (%)",
    note="Counts do not sum money across currencies. Categories follow transaction-count order in both panels; rates are descriptive, not causal.",
    height=900,
)

## Paid amounts in US dollars only

Load **all USD-paid label-1 rows**, plus USD-paid label-0 rows whose immutable database ID is divisible by 100. This is a deterministic systematic reference sample, **not a random sample**; source ordering may bias it. The inclusion rule deliberately changes class proportions, so do not estimate label prevalence from this frame.

Keep `amount_paid` as the original text and retain source hash and CSV record number. Exact decimal values support the descriptive table. Convert to floating point only for the logarithmic plot. No exchange-rate assumptions or cross-currency sums enter this section.

In [ ]:
from decimal import Decimal
USD_CURRENCY = "US Dollar"
REFERENCE_INTERVAL = 100
usd_reference = pd.read_sql_query(
    """SELECT id, source_sha256, source_row, amount_paid, is_laundering
       FROM transactions
       WHERE source_sha256 = ? AND payment_currency = ?
         AND (is_laundering = 1 OR (is_laundering = 0 AND id % ? = 0))
       ORDER BY id""",
    conn, params=(SOURCE_SHA, USD_CURRENCY, REFERENCE_INTERVAL),
)
assert usd_reference["amount_paid"].map(lambda value: isinstance(value, str)).all()
assert (usd_reference.loc[usd_reference["is_laundering"].eq(0), "id"] % REFERENCE_INTERVAL).eq(0).all()
usd_reference["amount_decimal"] = usd_reference["amount_paid"].map(Decimal)
assert usd_reference["amount_decimal"].map(lambda value: value.is_finite() and value > 0).all()
full_usd_counts = category_joint.loc[category_joint["payment_currency"].eq(USD_CURRENCY)].groupby("is_laundering")["transactions"].sum()
assert int(usd_reference["is_laundering"].eq(1).sum()) == int(full_usd_counts.get(1, 0))
assert len(usd_reference) < 100_000, "Unexpected reference size: revisit the notebook memory bound."

In [ ]:
usd_rows = []
for label, group in usd_reference.groupby("is_laundering", sort=True):
    values = sorted(group["amount_decimal"])
    middle = len(values) // 2
    median = values[middle] if len(values) % 2 else (values[middle - 1] + values[middle]) / Decimal(2)
    usd_rows.append({
        "Group": "Label 0: systematic reference" if label == 0 else "Label 1: all USD-paid rows",
        "Loaded rows": len(group), "Full USD group rows": int(full_usd_counts.get(label, 0)),
        "Minimum paid (USD)": str(values[0]), "Median paid (USD)": str(median), "Maximum paid (USD)": str(values[-1]),
    })
show_table(pd.DataFrame(usd_rows))
display(Markdown("Amount statistics describe the loaded rows only. The full-group count provides the denominator without implying the systematic reference is representative."))

In [ ]:
plot_amounts = usd_reference["amount_decimal"].map(float).to_numpy()
assert np.isfinite(plot_amounts).all() and (plot_amounts > 0).all()
log_amounts = np.log10(plot_amounts)
low, high = float(log_amounts.min()), float(log_amounts.max())
if low == high:
    low, high = low - 0.5, high + 0.5
bins = np.linspace(low, high, 36)
fig = go.Figure()
for label, colour, legend in [
    (0, GROUPS[0], "Label 0: systematic reference"),
    (1, GROUPS[1], "Label 1: all USD-paid rows"),
]:
    selected = log_amounts[usd_reference["is_laundering"].to_numpy() == label]
    if len(selected):
        bin_counts, bin_edges = np.histogram(selected, bins=bins)
        shares = bin_counts * (100.0 / len(selected))
        assert np.isclose(shares.sum(), 100.0)
        # Repeat bin edges to draw a histogram step, without converting plotted bin bounds into exact amounts.
        step_x = np.repeat(bin_edges, 2)
        step_y = np.r_[0.0, np.repeat(shares, 2), 0.0]
        hover_rows = np.repeat(np.column_stack([bin_counts, bin_edges[:-1], bin_edges[1:]]), 2, axis=0)
        hover_rows = np.vstack([hover_rows[0], hover_rows, hover_rows[-1]])
        fig.add_trace(go.Scatter(
            x=step_x, y=step_y, mode="lines", line=dict(color=colour, width=2),
            name=f"{legend} (n={len(selected):,})", customdata=hover_rows,
            hovertemplate="Approximate log₁₀ USD bin: %{customdata[1]:.3f}–%{customdata[2]:.3f}<br>Loaded rows in bin: %{customdata[0]:,}<br>Share of loaded group: %{y:.3f}%<extra>%{fullData.name}</extra>",
        ))
fig.update_xaxes(title_text="log₁₀(amount paid in USD); 2 = $100, 3 = $1,000, 4 = $10,000")
fig.update_yaxes(title_text="Share of loaded group in bin (%)", ticksuffix="%", rangemode="tozero")
finish_figure(
    fig, "eda-usd-paid-amount-shapes",
    title="USD amount shapes vary between the two loaded label groups",
    subtitle="USD-paid transactions only · all label-1 rows and deterministic label-0 ID-divisible-by-100 reference · 35 common bins",
    note="Each loaded group sums to 100%. The reference is nonrandom; exact decimal amounts remain in the table and source rows.",
    height=550,
)
display(Markdown("Each group's bin shares sum to 100%. The overlaid step histograms compare shape, not class prevalence, transaction totals, losses or stolen proceeds."))

### What this profile can and cannot establish

The fresh SQL summaries independently reproduce the audit's daily and category totals. They expose temporal imbalance, category composition and USD amount shapes. They do not validate ownership, employee relationships, shell-company status, invoice validity or embezzlement. The systematic amount reference can reflect source ordering; treat differences as exploratory observations, not a tested predictive result.

## Candidate-path analysis: overlap, timing, and reference-label coverage

This section reads the **completed scan**, not a new detection run. It examines every saved business → business → person candidate. The 905 paths overlap: they are **not 905 independent incidents**, and their transaction amounts must not be summed as losses. IBM name prefixes supply the entity roles; they do not establish employment, authorization, or shell-company status.

In [ ]:
import hashlib
import json
from decimal import Decimal, localcontext
from fractions import Fraction

candidate_path = REPORT_DIR / SCAN['candidate_output']['file_name']
candidate_bytes = candidate_path.read_bytes()
assert hashlib.sha256(candidate_bytes).hexdigest() == SCAN['candidate_output']['sha256'], 'Candidate JSONL hash mismatch'
assert len(candidate_bytes) == SCAN['candidate_output']['bytes'], 'Candidate JSONL size mismatch'
candidate_records = [json.loads(line) for line in candidate_bytes.splitlines() if line.strip()]
assert len(candidate_records) == SCAN['candidate_count'] == SCAN['candidate_output']['row_count'] == 905
assert SCAN['full_coverage'] is True and SCAN['truncated'] is False
transaction_sources = {source['sha256']: source for source in SCAN['sources']['transactions']}
account_source_hash = SCAN['sources']['accounts']['sha256']
for record in candidate_records:
    assert len(record['path']) == 2 and len(record['accounts']) == 3
    for edge in record['path']:
        assert edge['source_sha256'] in transaction_sources
        assert 2 <= edge['source_row'] <= transaction_sources[edge['source_sha256']]['row_count'] + 1
        assert 'is_laundering' not in edge, 'Labels must not be present in candidate selection records'
    assert all(account['source_sha256'] == account_source_hash for account in record['accounts'])
display(Markdown(f"**Verified saved output:** {len(candidate_records):,} paths; SHA-256 `{SCAN['candidate_output']['sha256']}`. No import or scan is rerun."))

In [ ]:
# Keep original amount strings and exact rational ratios. Decimal parses the
# amounts exactly; the 50-digit Decimal/float view below is only for display.
pair_rows = []
for record in candidate_records:
    first, second = record['path']
    origin, middle, person = record['accounts']
    ratio_exact = Fraction(Decimal(second['amount_paid'])) / Fraction(Decimal(first['amount_received']))
    lag_seconds = int((pd.Timestamp(second['timestamp']) - pd.Timestamp(first['timestamp'])).total_seconds())
    assert 0 < lag_seconds <= int(Decimal(SCAN['parameters']['window_hours']) * 3600)
    assert first['receiving_currency'] == second['payment_currency']
    assert Fraction(Decimal(SCAN['parameters']['min_ratio'])) <= ratio_exact <= Fraction(Decimal(SCAN['parameters']['max_ratio']))
    assert len({(a['bank'], a['account']) for a in record['accounts']}) == 3
    assert len({a['entity_id'] for a in record['accounts']}) == 3
    with localcontext() as decimal_context:
        decimal_context.prec = 50
        ratio_decimal = Decimal(ratio_exact.numerator) / Decimal(ratio_exact.denominator)
    pair_rows.append({
        'first_id': first['id'], 'second_id': second['id'],
        'first_source_sha256': first['source_sha256'], 'second_source_sha256': second['source_sha256'],
        'first_source_row': first['source_row'], 'second_source_row': second['source_row'],
        'first_source_ref': f"sha256:{first['source_sha256']}#record={first['source_row']}",
        'second_source_ref': f"sha256:{second['source_sha256']}#record={second['source_row']}",
        'first_timestamp': first['timestamp'], 'second_timestamp': second['timestamp'],
        'first_amount_paid_text': first['amount_paid'],
        'first_amount_received_text': first['amount_received'],
        'second_amount_paid_text': second['amount_paid'],
        'second_amount_received_text': second['amount_received'],
        'currency': first['receiving_currency'],
        'ratio_exact': ratio_exact, 'ratio_decimal': str(ratio_decimal), 'ratio': float(ratio_decimal),
        'lag_hours': lag_seconds / 3600,
        'origin_entity_id': origin['entity_id'], 'origin_entity_name': origin['entity_name'],
        'middle_entity_id': middle['entity_id'], 'middle_entity_name': middle['entity_name'],
        'person_entity_id': person['entity_id'], 'person_entity_name': person['entity_name'],
    })
pairs = pd.DataFrame(pair_rows)
assert not pairs.duplicated(['first_id', 'second_id']).any()
selected_ids = sorted(set(pairs['first_id']) | set(pairs['second_id']))
assert pairs['first_id'].nunique() == SCAN['unique_first_edges'] == 355
assert pairs['second_id'].nunique() == SCAN['unique_second_edges'] == 93
assert len(selected_ids) == SCAN['selected_edge_diagnostics']['total'] == 448
show_table(pd.DataFrame([
    {'Counting unit': 'Two-hop candidate paths', 'Count': len(pairs)},
    {'Counting unit': 'Distinct first-hop transactions', 'Count': pairs['first_id'].nunique()},
    {'Counting unit': 'Distinct second-hop transactions', 'Count': pairs['second_id'].nunique()},
    {'Counting unit': 'Distinct transactions in the union of both hops', 'Count': len(selected_ids)},
]))
funnel = pd.DataFrame({
    'Stage': ['Role, strict time, and currency checks', 'Inclusive adjacent-amount ratio check', 'Three distinct accounts and entities'],
    'Candidate pairs': [SCAN['examined']['time_currency_role_pairs'], SCAN['examined']['ratio_pass_pairs'], len(pairs)],
})
assert funnel['Candidate pairs'].tolist() == [19577, 932, 905]
assert SCAN['examined']['ownership_excluded_pairs'] == 27
show_table(funnel)
show_table(pd.DataFrame([
    {'Rule': 'Entity-role sequence', 'Configured value': 'business → business → person (derived IBM roles)'},
    {'Rule': 'Time order', 'Configured value': 'Second timestamp must be strictly later'},
    {'Rule': 'Total window', 'Configured value': SCAN['parameters']['window_hours'] + ' hours, inclusive upper bound'},
    {'Rule': 'Adjacent currency', 'Configured value': 'First received currency = second paid currency'},
    {'Rule': 'Amount ratio', 'Configured value': SCAN['parameters']['min_ratio'] + ' ≤ second paid / first received ≤ ' + SCAN['parameters']['max_ratio']},
    {'Rule': 'Identity', 'Configured value': 'Three distinct bank/account pairs and entity IDs'},
    {'Rule': 'Labels', 'Configured value': 'Excluded from selection and ranking'},
]))

The funnel counts **pairs**, not transactions: 19,577 pairs met the role, time, and currency conditions; 932 also met the amount ratio; 27 repeated an account or entity, leaving 905. These thresholds are engineering rules, not ACFE-prescribed proof of fraud. The amount ratio compares the first recipient's received amount with the next sender's paid amount **in the same currency**. We do not compare amounts across different currencies or attribute mixed funds to a particular origin.

In [ ]:
# Evaluate reference labels only after candidate IDs and all selection checks
# are fixed. Each read-only query binds at most 400 IDs.
label_chunks = []
for offset in range(0, len(selected_ids), 400):
    chunk = [int(value) for value in selected_ids[offset:offset + 400]]
    placeholders = ','.join('?' for _ in chunk)
    label_chunks.append(pd.read_sql_query(
        f'SELECT id,is_laundering FROM transactions WHERE id IN ({placeholders})',
        conn, params=chunk))
selected_labels = pd.concat(label_chunks, ignore_index=True)
assert len(selected_labels) == len(selected_ids) == 448
assert set(selected_labels['id']) == set(selected_ids)
assert selected_labels['id'].is_unique
assert set(selected_labels['is_laundering']) == {0}
label_by_id = selected_labels.set_index('id')['is_laundering']
pairs['first_reference_label'] = pairs['first_id'].map(label_by_id)
pairs['second_reference_label'] = pairs['second_id'].map(label_by_id)
assert int(selected_labels['is_laundering'].sum()) == SCAN['selected_edge_diagnostics']['laundering'] == 0
assert SCAN['ledger_label_diagnostics']['laundering'] == 5177
show_table(pd.DataFrame([
    {'Reference-label measure': 'Selected distinct transactions labeled laundering', 'Count': 0},
    {'Reference-label measure': 'All ledger transactions labeled laundering', 'Count': 5177},
    {'Reference-label measure': 'Selected distinct transactions labeled 0', 'Count': 448},
]))
display(Markdown('**0 of 5,177 laundering-labeled transactions appear in the selected edge union.** '
    'This is zero reference-label coverage for this narrow rule and indicates a poor fit to the dataset’s labeled laundering cases. '
    'It is **not accuracy**, proof that these candidates are benign, or an embezzlement evaluation: the dataset supplies no embezzlement ground truth. '
    'The label queries did not choose, remove, or rank any candidate.'))

In [ ]:
entity_counts = pd.DataFrame([
    {'Entity position': 'Origins', 'Distinct entities': pairs['origin_entity_id'].nunique()},
    {'Entity position': 'Intermediaries', 'Distinct entities': pairs['middle_entity_id'].nunique()},
    {'Entity position': 'Person recipients', 'Distinct entities': pairs['person_entity_id'].nunique()},
])
show_table(entity_counts)
recipient_concentration = (
    pairs.groupby(['person_entity_id', 'person_entity_name'], sort=True)
    .agg(candidate_paths=('first_id', 'size'), distinct_second_hops=('second_id', 'nunique'))
    .reset_index()
    .sort_values(['candidate_paths', 'person_entity_id'], ascending=[False, True], kind='stable')
)
recipient_concentration['share_of_paths_pct'] = recipient_concentration['candidate_paths'] / len(pairs) * 100
recipient_concentration['paths_per_second_hop'] = recipient_concentration['candidate_paths'] / recipient_concentration['distinct_second_hops']
assert int(recipient_concentration['candidate_paths'].sum()) == 905
assert int(recipient_concentration['distinct_second_hops'].sum()) == 93
show_table(recipient_concentration.head(8).rename(columns={
    'person_entity_id': 'Recipient entity ID', 'person_entity_name': 'Recipient name',
    'candidate_paths': 'Candidate paths (count)', 'distinct_second_hops': 'Distinct onward transactions (count)',
    'share_of_paths_pct': 'Share of paths (%)', 'paths_per_second_hop': 'Paths per onward transaction',
}).round({'Share of paths (%)': 2, 'Paths per onward transaction': 2}))

The charts expose two different properties of the saved candidates. The scatter shows the configured timing and amount-ratio region. The recipient comparison separates **path count** from **distinct onward-transaction count**: one outward payment can appear in several paths when several earlier inflows qualify. High path concentration therefore does not, by itself, measure the number or value of independent events. Recipients are ordered by path count with entity ID as the tie-breaker; labels play no part.

In [ ]:
fig = go.Figure(go.Scatter(
    x=pairs['lag_hours'], y=pairs['ratio'], mode='markers',
    marker=dict(symbol='square', size=6, color=FOCUS, line=dict(width=0)),
    customdata=pairs[['first_id','second_id','currency','person_entity_name','ratio_decimal']].to_numpy(),
    hovertemplate=('Lag: %{x:.2f} hours<br>Adjacent amount ratio: %{customdata[4]}'
                   '<br>First / second transaction: %{customdata[0]} / %{customdata[1]}'
                   '<br>Intermediate currency: %{customdata[2]}'
                   '<br>Recipient: %{customdata[3]}<extra>Candidate path</extra>'),
    showlegend=False))
fig.update_xaxes(title_text='Elapsed time between hops (hours)',
                 range=[0,float(Decimal(SCAN['parameters']['window_hours']))])
fig.update_yaxes(title_text='Second paid / first received amount (ratio)',
                 range=[float(Decimal(SCAN['parameters']['min_ratio']))-.01,
                        float(Decimal(SCAN['parameters']['max_ratio']))+.01])
for boundary in ['min_ratio','max_ratio']:
    fig.add_hline(y=float(Decimal(SCAN['parameters'][boundary])),
                  line_color=LINE,line_width=1,line_dash='dash')
finish_figure(fig,'candidate_lag_ratio',
    title='Candidate paths span the configured timing and amount bounds',
    subtitle=f"All {len(pairs):,} two-hop paths · hours and adjacent-amount ratios",
    note='Each square represents a path. Paths can share transactions; overlapping points are not jittered.',
    height=520)

In [ ]:
recipient_top = recipient_concentration.head(8).copy()
recipient_top_labels = recipient_top['person_entity_name'] + ' · ' + recipient_top['person_entity_id']
fig = go.Figure()
for recipient_column,recipient_label,recipient_color in [
    ('candidate_paths','Candidate paths',GROUPS[0]),
    ('distinct_second_hops','Distinct onward transactions',GROUPS[1])]:
    fig.add_trace(go.Bar(
        x=recipient_top[recipient_column],y=recipient_top_labels,orientation='h',
        name=recipient_label,marker=dict(color=recipient_color,line=dict(width=0)),
        text=recipient_top[recipient_column].map(lambda n:f'{n:,}'),textposition='outside',
        textfont=dict(color=TEXT),cliponaxis=False,
        customdata=recipient_top[['person_entity_id','person_entity_name']].to_numpy(),
        hovertemplate=('<b>%{customdata[1]}</b><br>Entity: %{customdata[0]}'
                       '<br>'+recipient_label+': %{x:,}<extra></extra>')))
fig.update_layout(barmode='group',bargap=.25,bargroupgap=.12)
fig.update_xaxes(title_text='Count — different units; do not add the series',
                 range=[0,float(recipient_top['candidate_paths'].max())*1.22],showgrid=False)
fig.update_yaxes(autorange='reversed',showgrid=False,
                 categoryorder='array',categoryarray=recipient_top_labels.tolist())
recipient_top_two_share = recipient_concentration.head(2)['candidate_paths'].sum()/len(pairs)
finish_figure(fig,'recipient_concentration',
    title=f'Two recipients account for {recipient_top_two_share:.1%} of candidate paths',
    subtitle='Top eight of 14 recipients · candidate paths versus distinct onward transactions (counts)',
    note='Paths overlap and are review leads. Reused onward transactions do not represent separate incidents or losses.',
    height=640)

In [ ]:
# Sample by source provenance order, never by target label or suspiciousness.
sample = pairs.sort_values(
    ['first_source_sha256', 'first_source_row', 'second_source_sha256', 'second_source_row'],
    kind='stable').head(5).copy()
sample_table = pd.DataFrame({
    'First source record': [transaction_sources[h]['file_name'] + '#record=' + str(r)
                            for h, r in zip(sample['first_source_sha256'], sample['first_source_row'])],
    'Second source record': [transaction_sources[h]['file_name'] + '#record=' + str(r)
                             for h, r in zip(sample['second_source_sha256'], sample['second_source_row'])],
    'Person recipient': sample['person_entity_name'].to_numpy(),
    'Lag (hours)': sample['lag_hours'].round(2).to_numpy(),
    'Amount ratio': sample['ratio'].round(5).to_numpy(),
    'Adjacent currency': sample['currency'].to_numpy(),
    'Reference labels (first / second)': [f'{a} / {b}' for a, b in
                                         zip(sample['first_reference_label'], sample['second_reference_label'])],
})
show_table(sample_table)
display(Markdown('Source record numbers include the CSV header as record 1. Full SHA-256 references and '
    'original amount strings remain in `pairs`; the manifest above identifies the source files. '
    '**Label 0 is a dataset reference label, not a finding of innocence. These 905 overlapping paths are review leads, '
    'not independent incidents, proven embezzlement, or calculated losses.**'))

## Advanced temporal analysis: distribution shift and category adjustment

**Distribution shift** means that the mix of observed transactions changes over time. We measure changes in paid currency, payment format, and their joint categories, using every ledger row. **Jensen–Shannon distance** compares two category-share distributions: 0 means identical shares; 1 means completely separate support when logarithms use base 2. Zero-count categories remain valid; no smoothing constant or arbitrary alarm threshold is added. The NumPy implementation below follows the [SciPy definition](https://docs.scipy.org/doc/scipy/reference/generated/scipy.spatial.distance.jensenshannon.html).

The main-window reference pools September 1–10. The daily chart compares each day with that pooled reference, which includes the day itself during the main window. This is a retrospective description, not a future-only monitoring simulation. Label-conditioned distances are separate retrospective diagnostics; labels never enter candidate features or ranking. Distances describe this finite synthetic ledger: they are not fraud probabilities, significance tests, or evidence that the simulator represents a bank's future traffic.

The [ACFE analytics test library](https://www.acfe.com/fraud-resources/fraud-risk-tools---coso/anti-fraud-data-analytics-tests) includes period comparisons and expense stratification within tests for specific fraud risks. We apply that general principle here. Our currency/format stratification, distance metric, and adjustment formulas are analytical choices, not ACFE-prescribed methods, thresholds, or proof of misconduct.

In [ ]:
# This compact result aggregates the entire ledger, without sampling transaction rows.
adv_stats_grouped = pd.read_sql_query(
    """SELECT substr(timestamp,1,10) AS date, payment_currency, payment_format,
              is_laundering, COUNT(*) AS n
       FROM transactions WHERE source_sha256 = ?
       GROUP BY substr(timestamp,1,10),payment_currency,payment_format,is_laundering
       ORDER BY date,payment_currency,payment_format,is_laundering""",
    conn, params=(AUDIT['sources']['transactions']['sha256'],))
assert int(adv_stats_grouped['n'].sum()) == AUDIT['transactions']['rows']
assert int(adv_stats_grouped.loc[adv_stats_grouped['is_laundering'].eq(1),'n'].sum()) == AUDIT['transactions']['label_counts']['1']
adv_stats_grouped['period'] = np.where(adv_stats_grouped['date'] < '2022-09-11', 'Main', 'Tail')
adv_stats_dimensions = {
    'Paid currency': ['payment_currency'],
    'Payment format': ['payment_format'],
    'Currency × format': ['payment_currency','payment_format'],
}
def adv_stats_js_distance(left, right):
    """Base-2 Jensen–Shannon distance; aligns category support explicitly."""
    aligned = pd.concat([left.rename('left'),right.rename('right')],axis=1).fillna(0)
    values = aligned.to_numpy(dtype=float)
    assert (values >= 0).all() and (values.sum(axis=0) > 0).all()
    p, q = (values / values.sum(axis=0)).T
    midpoint = (p+q)/2
    p_nonzero, q_nonzero = p>0, q>0
    divergence = 0.5*(np.sum(p[p_nonzero]*np.log2(p[p_nonzero]/midpoint[p_nonzero])) +
                      np.sum(q[q_nonzero]*np.log2(q[q_nonzero]/midpoint[q_nonzero])))
    return float(np.sqrt(np.clip(divergence,0,1)))
# Boundary cases make the distance's units and treatment of zero categories explicit.
assert adv_stats_js_distance(pd.Series([1,0]),pd.Series([0,1])) == 1
assert adv_stats_js_distance(pd.Series([1,2]),pd.Series([2,4])) == 0
adv_stats_distance_rows=[]
for adv_stats_population,adv_stats_label in [('All rows',None),('Label 0 (retrospective)',0),('Label 1 (retrospective)',1)]:
    adv_stats_population_frame = (adv_stats_grouped if adv_stats_label is None else
                                adv_stats_grouped.loc[adv_stats_grouped['is_laundering'].eq(adv_stats_label)])
    adv_stats_main = adv_stats_population_frame.loc[adv_stats_population_frame['period'].eq('Main')]
    adv_stats_tail = adv_stats_population_frame.loc[adv_stats_population_frame['period'].eq('Tail')]
    for adv_stats_dimension,adv_stats_columns in adv_stats_dimensions.items():
        adv_stats_left = adv_stats_main.groupby(adv_stats_columns)['n'].sum()
        adv_stats_right = adv_stats_tail.groupby(adv_stats_columns)['n'].sum()
        adv_stats_distance = adv_stats_js_distance(adv_stats_left,adv_stats_right)
        assert np.isclose(adv_stats_distance,adv_stats_js_distance(adv_stats_right,adv_stats_left))
        adv_stats_distance_rows.append({'Population':adv_stats_population,'Category distribution':adv_stats_dimension,
            'Main rows':int(adv_stats_main['n'].sum()),'Tail rows':int(adv_stats_tail['n'].sum()),
            'Jensen–Shannon distance (0–1)':adv_stats_distance})
adv_stats_distances = pd.DataFrame(adv_stats_distance_rows)
show_table(adv_stats_distances,formats={'Jensen–Shannon distance (0–1)':'{:.4f}'})

In [ ]:
adv_stats_daily_distances=[]
for adv_stats_dimension,adv_stats_columns in adv_stats_dimensions.items():
    adv_stats_baseline = adv_stats_grouped.loc[adv_stats_grouped['period'].eq('Main')].groupby(adv_stats_columns)['n'].sum()
    for adv_stats_date,adv_stats_day in adv_stats_grouped.groupby('date',sort=True):
        adv_stats_observed = adv_stats_day.groupby(adv_stats_columns)['n'].sum()
        adv_stats_daily_distances.append({'date':pd.Timestamp(adv_stats_date),'dimension':adv_stats_dimension,
            'distance':adv_stats_js_distance(adv_stats_baseline,adv_stats_observed),'rows':int(adv_stats_day['n'].sum())})
adv_stats_daily_distances = pd.DataFrame(adv_stats_daily_distances)
adv_stats_fig = go.Figure()
for adv_stats_series_index,(adv_stats_dimension,adv_stats_color) in enumerate(zip(adv_stats_dimensions,GROUPS)):
    adv_stats_plot = adv_stats_daily_distances.loc[adv_stats_daily_distances['dimension'].eq(adv_stats_dimension)]
    adv_stats_fig.add_trace(go.Scatter(
        x=adv_stats_plot['date'],y=adv_stats_plot['distance'],mode='lines',
        line=dict(color=adv_stats_color,width=2),name=adv_stats_dimension,
        customdata=adv_stats_plot[['rows']].to_numpy(),
        hovertemplate='%{x|%d %b %Y}<br>Distance: %{y:.4f}<br>Transactions: %{customdata[0]:,}<extra>%{fullData.name}</extra>'))
    adv_stats_endpoint = adv_stats_plot.iloc[-1]
    adv_stats_fig.add_annotation(x=adv_stats_endpoint['date'],y=adv_stats_endpoint['distance'],
        text=f"{adv_stats_endpoint['distance']:.3f}",showarrow=False,xanchor='right',
        xshift=-5,yshift=10+adv_stats_series_index*9,font=dict(color=TEXT,size=11))
adv_stats_dates = sorted(adv_stats_daily_distances['date'].unique())
adv_stats_tick_positions = np.unique(np.linspace(0,len(adv_stats_dates)-1,min(7,len(adv_stats_dates)),dtype=int))
adv_stats_fig.update_xaxes(tickmode='array',tickvals=[adv_stats_dates[i] for i in adv_stats_tick_positions],
    tickformat='%d %b',range=[pd.Timestamp(adv_stats_dates[0])-pd.Timedelta(days=0.6),
                           pd.Timestamp(adv_stats_dates[-1])+pd.Timedelta(days=0.6)],
    title_text='Source transaction date')
adv_stats_fig.update_yaxes(range=[0,1],title_text='Jensen–Shannon distance (0–1)')
adv_stats_fig.add_annotation(x=pd.Timestamp('2022-09-11'),y=0.99,text='Tail begins: 11 Sep',
    showarrow=False,xanchor='left',yanchor='top',font=dict(color=TEXT,size=11))
finish_figure(adv_stats_fig,'advanced-distribution-shift',
    title='Late transactions have a different currency and payment-format mix',
    subtitle='Distance to pooled 1–10 Sep category shares · all ledger rows',
    note='Main-window daily volumes: 207,382–1,114,921; tail: 11–396. Main days are included in their own reference. No alarm threshold.',height=500)
display(Markdown('The main-window days contain 207,382–1,114,921 rows; the tail days contain only 11–396. '
    'Small daily denominators make the tail composition sensitive to individual rows. '
    'The chart reports actual ledger differences, without an alert cutoff or a claim of statistical significance.'))

### How much of the label-rate change comes from a different category mix?

A pooled label rate can change because category shares change, because rates within categories change, or both. **Direct standardization** holds the main-window category label rates fixed and applies them to the observed tail category shares. We do this first by payment format and then by the joint paid-currency × payment-format category.

For categories $c$, the standardized rate is $r_{std}=\sum_c w_{tail,c}r_{main,c}$. The exact identity is

$$r_{tail}-r_{main}=(r_{std}-r_{main})+(r_{tail}-r_{std}).$$

The first term describes the change associated with replacing category weights while holding main rates fixed; the second is the remaining difference. This decomposition depends on that reference choice. It is a descriptive arithmetic identity, **not a causal explanation or a model predicting fraudulent payments**. Categories present only in the main window receive zero tail weight; every tail category must have a main-window rate. The code checks that support before calculating the adjustment. A noninteger expected count is a weighted reference calculation, not a new label assigned to any row.

Method reference: [NCHS direct-standardization notes](https://www.cdc.gov/Nchs/data/statnt/statnt06rv.pdf) describe the same weighted-rate arithmetic; this notebook applies it to transaction categories.

In [ ]:
adv_stats_period_counts = adv_stats_grouped.groupby('period')['n'].sum()
adv_stats_positive_counts = adv_stats_grouped.loc[adv_stats_grouped['is_laundering'].eq(1)].groupby('period')['n'].sum()
adv_stats_main_rate = adv_stats_positive_counts['Main']/adv_stats_period_counts['Main']
adv_stats_tail_rate = adv_stats_positive_counts['Tail']/adv_stats_period_counts['Tail']
adv_stats_adjustments=[]
for adv_stats_name,adv_stats_columns in [('Payment format',['payment_format']),('Currency × format',['payment_currency','payment_format'])]:
    adv_stats_totals = adv_stats_grouped.groupby(['period']+adv_stats_columns)['n'].sum()
    adv_stats_positives = adv_stats_grouped.loc[adv_stats_grouped['is_laundering'].eq(1)].groupby(['period']+adv_stats_columns)['n'].sum()
    adv_stats_strata = pd.DataFrame({'rows':adv_stats_totals,'positive':adv_stats_positives}).fillna(0)
    adv_stats_main_strata = adv_stats_strata.loc['Main'].copy()
    adv_stats_tail_strata = adv_stats_strata.loc['Tail'].copy()
    assert adv_stats_tail_strata.index.isin(adv_stats_main_strata.index).all(), 'Tail category lacks a main-window reference rate.'
    adv_stats_main_strata['rate'] = adv_stats_main_strata['positive']/adv_stats_main_strata['rows']
    adv_stats_tail_reference = adv_stats_main_strata['rate'].reindex(adv_stats_tail_strata.index)
    assert adv_stats_tail_reference.notna().all()
    adv_stats_expected_tail = float((adv_stats_tail_strata['rows']*adv_stats_tail_reference).sum())
    adv_stats_standardized_rate = adv_stats_expected_tail/adv_stats_period_counts['Tail']
    adv_stats_mix_pp = 100*(adv_stats_standardized_rate-adv_stats_main_rate)
    adv_stats_remaining_pp = 100*(adv_stats_tail_rate-adv_stats_standardized_rate)
    assert np.isclose(adv_stats_mix_pp+adv_stats_remaining_pp,100*(adv_stats_tail_rate-adv_stats_main_rate),rtol=1e-12)
    adv_stats_adjustments.append({'Adjustment categories':adv_stats_name,
        'Main observed rate (%)':100*adv_stats_main_rate,
        'Tail mix × main rates (%)':100*adv_stats_standardized_rate,
        'Tail observed rate (%)':100*adv_stats_tail_rate,
        'Category-mix component (percentage points)':adv_stats_mix_pp,
        'Remaining component (percentage points)':adv_stats_remaining_pp,
        'Expected tail label-1 count at main rates':adv_stats_expected_tail})
adv_stats_adjustments=pd.DataFrame(adv_stats_adjustments)
show_table(adv_stats_adjustments.drop(columns=['Main observed rate (%)','Tail observed rate (%)']),
           formats={c:'{:.4f}' for c in adv_stats_adjustments.select_dtypes(include='number').columns})
adv_stats_tail_formats = adv_stats_grouped.loc[adv_stats_grouped['period'].eq('Tail')].groupby('payment_format')['n'].sum()
adv_stats_tail_row_share = adv_stats_period_counts['Tail']/adv_stats_period_counts.sum()
adv_stats_tail_positive_share = adv_stats_positive_counts['Tail']/adv_stats_positive_counts.sum()
display(Markdown(f"The main-window label rate is {100*adv_stats_main_rate:.4f}%, compared with {100*adv_stats_tail_rate:.4f}% in the tail. "
    f"**The tail contributes {adv_stats_tail_row_share:.4%} of all rows but {adv_stats_tail_positive_share:.2%} of label-1 rows.** "
    f"Its {int(adv_stats_period_counts['Tail']):,} transactions use {len(adv_stats_tail_formats)} payment format(s): "
    + ', '.join(f'{k} ({int(v):,})' for k,v in adv_stats_tail_formats.items()) + '. '
    f"Holding main-window format rates fixed gives {adv_stats_adjustments.iloc[0]['Expected tail label-1 count at main rates']:.2f} "
    f"expected tail positives, versus {int(adv_stats_positive_counts['Tail']):,} observed. "
    'Changing the payment-format mix alone therefore does not reproduce the tail label rate. '
    'The remaining component describes changes within observed categories and any unmeasured composition; it does not establish their cause.'))

**Implication for evaluation:** report main-window and tail results separately and inspect category support before applying a model across the boundary. The sharp shift affects even the label-0 population, so a high overall distance cannot be read as a fraud signal. Do not select dates, features, or thresholds from these retrospective labels and then present performance on those same rows as an independent test. Reserve a genuinely later or separately generated dataset for prospective evaluation.

The ledger is a complete synthetic population for this analysis, and transactions can share accounts, entities, and laundering attempts. Treating millions of rows as independent random draws would create misleadingly precise inferential results. No confidence intervals or hypothesis-test p-values are reported. The tail may reflect simulator scheduling or dataset construction; this notebook cannot identify that mechanism. See the [IBM dataset paper](https://arxiv.org/html/2306.16424v3) for simulator context, and retain the earlier evidence gaps for any embezzlement investigation.

## Advanced network structure: account and entity views

A **directed edge** connects a sender to a recipient. We collapse repeated payments to the same directed pair when counting counterparties, and remove self-links from degree and reciprocity calculations. **Out-degree** counts distinct recipients; **in-degree** counts distinct senders. **Reciprocity** is the share of directed pairs whose reverse pair also appears somewhere in this snapshot. It is not a time-ordered refund or a laundering cycle.

We compare account keys `(normalized bank ID, account number)` with the publisher's entity IDs. Grouping accounts can reveal connections hidden by account fragmentation; it can also merge routine internal transfers. Every ledger row enters the count-based analysis. We stream endpoint identifiers and retain compact integer arrays, without building a Python graph or joining the ledger to itself. No monetary values or currencies are added.

ACFE recommends mapping documented connections and checking supporting business records; our degree, reciprocity and concentration measures are exploratory engineering choices, not ACFE fraud thresholds. Sources: [ACFE shell-company investigation guidance](https://www.acfe.com/fraud-resources/shell-companies), [ACFE procurement link-analysis discussion](https://www.acfe.com/fraud-magazine/all-issues/issue/article?s=2023-janfeb-procurement-integrity-breaches-data-analytics), and [NetworkX reciprocity definition](https://networkx.org/documentation/stable/reference/algorithms/generated/networkx.algorithms.reciprocity.reciprocity.html).

In [ ]:
import csv
from embezzlement_analysis.ingest import normalize_bank

adv_net_lookup = {}
adv_net_entity_lookup = {}
adv_net_account_entity = []
with (ROOT / 'data/raw/full-hi-small/HI-Small_accounts.csv').open(newline='') as adv_net_handle:
    for adv_net_record in csv.DictReader(adv_net_handle):
        adv_net_key = (normalize_bank(adv_net_record['Bank ID']), adv_net_record['Account Number'])
        assert adv_net_key not in adv_net_lookup, 'Account metadata must join unambiguously.'
        adv_net_lookup[adv_net_key] = len(adv_net_lookup)
        adv_net_entity = adv_net_record['Entity ID']
        if adv_net_entity not in adv_net_entity_lookup:
            adv_net_entity_lookup[adv_net_entity] = len(adv_net_entity_lookup)
        adv_net_account_entity.append(adv_net_entity_lookup[adv_net_entity])
adv_net_account_entity = np.asarray(adv_net_account_entity, dtype=np.int32)
adv_net_total = int(AUDIT['transactions']['rows'])
adv_net_endpoints = np.empty((adv_net_total, 2), dtype=np.int32)
adv_net_cursor = conn.execute('SELECT from_bank,from_account,to_bank,to_account FROM transactions WHERE source_sha256=?',
                              (AUDIT['sources']['transactions']['sha256'],))
adv_net_offset = 0
while True:
    adv_net_chunk = adv_net_cursor.fetchmany(100_000)
    if not adv_net_chunk:
        break
    adv_net_size = len(adv_net_chunk)
    adv_net_endpoints[adv_net_offset:adv_net_offset+adv_net_size, 0] = np.fromiter(
        (adv_net_lookup[(adv_net_row[0], adv_net_row[1])] for adv_net_row in adv_net_chunk), dtype=np.int32, count=adv_net_size)
    adv_net_endpoints[adv_net_offset:adv_net_offset+adv_net_size, 1] = np.fromiter(
        (adv_net_lookup[(adv_net_row[2], adv_net_row[3])] for adv_net_row in adv_net_chunk), dtype=np.int32, count=adv_net_size)
    adv_net_offset += adv_net_size
assert adv_net_offset == adv_net_total
adv_net_entity_endpoints = adv_net_account_entity[adv_net_endpoints]
assert len(adv_net_lookup) == AUDIT['accounts']['rows']
del adv_net_chunk, adv_net_cursor, adv_net_lookup, adv_net_entity_lookup
print(f'Network endpoint rows: {adv_net_total:,}; compact account + entity arrays: {(adv_net_endpoints.nbytes + adv_net_entity_endpoints.nbytes)/1024**2:.1f} MiB.')

In [ ]:
def adv_net_profile(adv_net_edges, adv_net_node_count, adv_net_level):
    adv_net_sender, adv_net_recipient = adv_net_edges[:, 0], adv_net_edges[:, 1]
    adv_net_out_rows = np.bincount(adv_net_sender, minlength=adv_net_node_count)
    adv_net_in_rows = np.bincount(adv_net_recipient, minlength=adv_net_node_count)
    adv_net_active = adv_net_out_rows + adv_net_in_rows > 0
    adv_net_nonself = adv_net_sender != adv_net_recipient
    # Integer encoding is reversible and safe here: node_count squared is < 2**63.
    assert adv_net_node_count ** 2 < np.iinfo(np.int64).max
    adv_net_codes = np.unique(adv_net_sender[adv_net_nonself].astype(np.int64)*adv_net_node_count + adv_net_recipient[adv_net_nonself])
    adv_net_from = adv_net_codes // adv_net_node_count
    adv_net_to = adv_net_codes % adv_net_node_count
    adv_net_out_degree = np.bincount(adv_net_from, minlength=adv_net_node_count)
    adv_net_in_degree = np.bincount(adv_net_to, minlength=adv_net_node_count)
    adv_net_reverse = adv_net_to*adv_net_node_count + adv_net_from
    adv_net_positions = np.searchsorted(adv_net_codes, adv_net_reverse)
    adv_net_within = adv_net_positions < len(adv_net_codes)
    adv_net_reciprocal = int(np.count_nonzero(adv_net_codes[adv_net_positions[adv_net_within]] == adv_net_reverse[adv_net_within]))
    assert adv_net_out_rows.sum() == adv_net_in_rows.sum() == adv_net_total
    assert adv_net_out_degree.sum() == adv_net_in_degree.sum() == len(adv_net_codes)
    assert adv_net_reciprocal % 2 == 0
    adv_net_summary = {'Network level':adv_net_level, 'Observed nodes':int(adv_net_active.sum()),
        'Distinct directed non-self pairs':len(adv_net_codes),
        'Reciprocal directed pairs':adv_net_reciprocal,
        'Reciprocity':adv_net_reciprocal/len(adv_net_codes) if len(adv_net_codes) else np.nan,
        'Self-link transaction rows':int((~adv_net_nonself).sum()),
        'Maximum in-degree':int(adv_net_in_degree.max()), 'Maximum out-degree':int(adv_net_out_degree.max())}
    return {'summary':adv_net_summary, 'active':adv_net_active, 'out_rows':adv_net_out_rows,
            'in_rows':adv_net_in_rows, 'out_degree':adv_net_out_degree, 'in_degree':adv_net_in_degree}

adv_net_account_profile = adv_net_profile(adv_net_endpoints, len(adv_net_account_entity), 'Bank/account')
adv_net_entity_profile = adv_net_profile(adv_net_entity_endpoints, int(adv_net_account_entity.max())+1, 'Publisher entity')
adv_net_profiles = [adv_net_account_profile, adv_net_entity_profile]
adv_net_summary = pd.DataFrame([adv_net_p['summary'] for adv_net_p in adv_net_profiles])
show_table(adv_net_summary, formats={'Reciprocity':'{:.2%}'})
adv_net_same_account = adv_net_endpoints[:,0] == adv_net_endpoints[:,1]
adv_net_same_entity = adv_net_entity_endpoints[:,0] == adv_net_entity_endpoints[:,1]
assert not np.any(adv_net_same_account & ~adv_net_same_entity)
adv_net_same_entity_other_account = int(np.count_nonzero(adv_net_same_entity & ~adv_net_same_account))
display(Markdown(f'**{adv_net_same_entity_other_account:,} transaction rows connect different accounts of the same publisher entity.** '
                 'Entity-level self-links also include same-account rows. Reverse-pair matches ignore time, amount and currency; they do not prove round trips or common funds.'))

In [ ]:
# For degrees [1,5], P(degree >= x) is 0.5 throughout (1,5].
# Plotly 'vh' draws vertical first at x=1, then horizontal to x=5.
# This trace fixture checks the exact coordinates and rendering contract.
adv_net_fixture_trace = go.Scatter(x=[1,5],y=[1,0.5],mode='lines',line=dict(shape='vh',width=2))
adv_net_fixture_spec = adv_net_fixture_trace.to_plotly_json()
assert list(adv_net_fixture_spec['x']) == [1,5]
assert list(adv_net_fixture_spec['y']) == [1,0.5]
assert adv_net_fixture_spec['line']['shape'] == 'vh'
assert adv_net_fixture_spec['mode'] == 'lines'
adv_net_fig = make_subplots(rows=1,cols=2,horizontal_spacing=0.17,
    subplot_titles=['Incoming counterparties','Outgoing counterparties'])
for adv_net_column,adv_net_direction in enumerate(['in','out'],start=1):
    for adv_net_series_index,(adv_net_profile_result,adv_net_color,adv_net_label) in enumerate(zip(
            adv_net_profiles,GROUPS,['Bank/account','Publisher entity'])):
        adv_net_degree = adv_net_profile_result[adv_net_direction+'_degree'][adv_net_profile_result['active']]
        adv_net_values,adv_net_frequencies = np.unique(adv_net_degree,return_counts=True)
        adv_net_survival = np.cumsum(adv_net_frequencies[::-1])[::-1]/len(adv_net_degree)
        adv_net_positive = adv_net_values>0
        adv_net_plot_x,adv_net_plot_y = adv_net_values[adv_net_positive],adv_net_survival[adv_net_positive]
        adv_net_fig.add_trace(go.Scatter(x=adv_net_plot_x,y=adv_net_plot_y,mode='lines',
            line=dict(color=adv_net_color,width=2,shape='vh'),name=adv_net_label,
            legendgroup=adv_net_label,showlegend=adv_net_column==1,
            hovertemplate='Degree: %{x:,}<br>Share of observed nodes: %{y:.6%}<extra>%{fullData.name}</extra>'),
            row=1,col=adv_net_column)
        adv_net_fig.add_annotation(x=float(adv_net_plot_x[-1]),y=float(adv_net_plot_y[-1]),
            text=f"{adv_net_plot_y[-1]:.5%}",showarrow=False,xanchor='right',xshift=-4,
            yshift=10+adv_net_series_index*10,font=dict(color=TEXT,size=10),row=1,col=adv_net_column)
    adv_net_fig.update_xaxes(type='log',title_text='Distinct '+('senders' if adv_net_direction=='in' else 'recipients')+' (log scale)',
        row=1,col=adv_net_column)
    adv_net_fig.update_yaxes(type='log',title_text='Share with at least this degree (log scale)',
        tickformat='.3~%',row=1,col=adv_net_column)
finish_figure(adv_net_fig,'advanced-network-degree-survival',
    title='Account and entity views reveal different counterparty structures',
    subtitle='Distinct non-self counterparties · share of all observed nodes · both axes logarithmic',
    note='Exact empirical survival curves. The denominator includes zero-degree nodes; zero is outside the log x-axis. Reverse links ignore time.',height=540)
display(Markdown('The denominator includes every observed node, including zero-degree nodes. Zero lies outside the logarithmic x-axis. '
                 'Self-links are excluded. Curves describe the complete snapshot without inferring temporal paths or fraud risk.'))

### Concentration and counting-unit sensitivity

The curve below orders observed nodes by **outgoing transaction-row count**, from lowest to highest. It compares the cumulative node share with the cumulative share of outgoing rows. **Concentration includes self-link transaction rows**, whereas degree and reciprocity exclude self-links. At entity level, these self-links include transfers between different accounts belonging to the same publisher entity. A highly uneven curve can reflect routine payment hubs or the simulator's design. The top 1% statistic rounds the node count upward and includes nodes with zero outgoing rows in its denominator. It is a descriptive concentration measure, not a financial loss or suspicion score. Aggregating multiple accounts changes both the population and the ranking; compare the two levels with that change in mind.

In [ ]:
adv_net_fig = go.Figure()
adv_net_concentration_rows=[]
adv_net_display_counts=[]
for adv_net_series_index,(adv_net_profile_result,adv_net_color,adv_net_label) in enumerate(zip(
        adv_net_profiles,GROUPS,['Bank/account','Publisher entity'])):
    adv_net_activity = np.sort(adv_net_profile_result['out_rows'][adv_net_profile_result['active']])
    adv_net_curve_x = np.arange(len(adv_net_activity)+1)/len(adv_net_activity)
    adv_net_curve_y = np.r_[0,np.cumsum(adv_net_activity,dtype=np.int64)]/adv_net_total
    adv_net_top_n = max(1,int(np.ceil(0.01*len(adv_net_activity))))
    # Only display coordinates are thinned. Every statistic uses all observed nodes.
    # Deterministic rank grid includes both endpoints, the top-1% boundary and dense upper tails.
    adv_net_node_n = len(adv_net_activity)
    adv_net_display_ranks = np.unique(np.r_[
        np.linspace(0,adv_net_node_n,min(1200,adv_net_node_n+1),dtype=np.int64),
        adv_net_node_n-np.geomspace(1,adv_net_node_n,min(600,adv_net_node_n)).astype(np.int64),
        np.arange(min(10,adv_net_node_n)+1),
        adv_net_node_n-np.arange(min(10,adv_net_node_n)+1),
        [0,adv_net_node_n-adv_net_top_n,adv_net_node_n]])
    assert adv_net_display_ranks[0] == 0 and adv_net_display_ranks[-1] == adv_net_node_n
    assert adv_net_node_n-adv_net_top_n in adv_net_display_ranks
    assert len(adv_net_display_ranks) <= 2000
    adv_net_display_counts.append((adv_net_label,len(adv_net_display_ranks),adv_net_node_n+1))
    adv_net_fig.add_trace(go.Scatter(x=adv_net_curve_x[adv_net_display_ranks],
        y=adv_net_curve_y[adv_net_display_ranks],mode='lines',
        line=dict(color=adv_net_color,width=2),name=adv_net_label,
        customdata=adv_net_display_ranks,
        hovertemplate='Cumulative node share: %{x:.3%}<br>Outgoing row share: %{y:.3%}<br>Rank: %{customdata:,}<extra>%{fullData.name}</extra>'))
    adv_net_fig.add_annotation(x=1,y=1,text=f'{adv_net_label}: 100%',showarrow=False,
        xanchor='right',xshift=-5,yshift=13 if adv_net_series_index==0 else -15,
        font=dict(color=TEXT,size=11))
    adv_net_concentration_rows.append({'Network level':adv_net_label, 'Observed nodes':len(adv_net_activity),
        'Nodes in top 1% (rounded up)':adv_net_top_n,
        'Outgoing rows from top 1%':int(adv_net_activity[-adv_net_top_n:].sum()),
        'Top 1% share of outgoing rows':float(adv_net_activity[-adv_net_top_n:].sum()/adv_net_total),
        'Nodes with both incoming and outgoing non-self counterparties':int(np.count_nonzero(
            (adv_net_profile_result['in_degree']>0)&(adv_net_profile_result['out_degree']>0)))})
    assert np.isclose(adv_net_curve_y[-1],1)
adv_net_fig.add_trace(go.Scatter(x=[0,1],y=[0,1],mode='lines',line=dict(color=REST,width=2,dash='dash'),
    name='Equal outgoing activity',hovertemplate='Node share = row share: %{x:.0%}<extra>Equal activity reference</extra>'))
adv_net_fig.update_xaxes(title_text='Cumulative share of observed nodes (lowest outgoing count first)',range=[0,1],tickformat='.0%')
adv_net_fig.update_yaxes(title_text='Cumulative share of outgoing transaction rows',range=[0,1.05],tickformat='.0%')
adv_net_display_note = '; '.join(f'{label}: {shown:,} of {total:,} rank coordinates' for label,shown,total in adv_net_display_counts)
finish_figure(adv_net_fig,'advanced-network-activity-concentration',
    title='Account and entity aggregation change activity concentration',
    subtitle='Outgoing transaction-row counts · includes self-links · all observed nodes',
    note='Display uses a deterministic grid of ≤2,000 points per curve, with denser upper tails. All summaries use exact full counts.',height=540)
display(Markdown('**Display downsampling:** '+adv_net_display_note+'. '
    'Plotted coordinates use exact cumulative totals at retained ranks; connecting segments approximate omitted ranks. '
    'Both endpoints and the top-1% boundary are retained. No transaction rows are sampled for network summaries.'))
adv_net_concentration = pd.DataFrame(adv_net_concentration_rows)
show_table(adv_net_concentration,formats={'Top 1% share of outgoing rows':'{:.2%}'})
# Keep node-level profiles for follow-up, but release full transaction endpoint arrays.
del adv_net_endpoints,adv_net_entity_endpoints,adv_net_same_account,adv_net_same_entity

**Interpretation limits.** A node with many counterparties is a hub in these supplied records, not an identified intermediary in an illicit scheme. A node with both incoming and outgoing counterparties need not receive before paying. Full-period degree and reciprocity would leak future information if reused as earlier-time predictive features; future models must construct them using only records available before each prediction. The publisher's entity ID groups synthetic accounts; it does not independently verify beneficial ownership or an employee relationship. Structural motifs cannot establish a ghost entity, shell status, the same funds moving onward, or embezzlement. Payment processors, treasury accounts, routine settlements and ordinary corporate transfers offer competing explanations. Establishing a case needs invoices, goods/services evidence, authorization history and sourced ownership/employee links.

## Chronological account-day anomaly ranking

**Question:** which outgoing account-day profiles in a later period differ most from earlier activity? An **account-day** means one sender bank/account on one calendar date, scored after that day closes. This is a global peer comparison, not an account-specific behavior model or real-time alert.

Use every **USD → USD** transaction during September 1–10, aggregate in SQLite, fit on September 1–7 and score September 8–10. Exclude the shifted September 11–18 tail. Freeze six features, seed 42, 100 trees, 256 observations per tree and a **top 1% review budget** before joining labels to these ranks. This is an exploratory analysis, not a preregistered study; earlier notebook sections already examined AML labels. Training is unlabeled and may contain illicit activity. Neither identifiers, dates nor AML labels enter the model.

[ACFE analytics tests](https://www.acfe.com/fraud-resources/fraud-risk-tools---coso/anti-fraud-data-analytics-tests) motivate targeted exception analysis; these models complement the transaction-path rule. [Isolation Forest](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.IsolationForest.html) isolates unusual feature combinations through random tree splits. Its scores express relative unusualness, **not fraud probability**. A transparent upper-tail baseline provides a comparison; no model or threshold is selected using labels.

In [ ]:
import time
from sklearn.ensemble import IsolationForest
from importlib.metadata import version as adv_anom_version
adv_anom_started = time.perf_counter()
adv_anom_population = pd.read_sql_query("""
    SELECT COUNT(*) AS all_rows,
           SUM(timestamp >= '2022-09-01' AND timestamp < '2022-09-11') AS main_rows,
           SUM(timestamp >= '2022-09-01' AND timestamp < '2022-09-11'
               AND payment_currency='US Dollar') AS main_paid_usd_rows,
           SUM(timestamp >= '2022-09-01' AND timestamp < '2022-09-11'
               AND payment_currency='US Dollar' AND receiving_currency='US Dollar') AS eligible_rows,
           SUM(timestamp >= '2022-09-01' AND timestamp < '2022-09-11'
               AND payment_currency='US Dollar' AND receiving_currency!='US Dollar') AS excluded_usd_fx_rows
    FROM transactions
""", conn).iloc[0].astype('int64')
# Numeric casts here support exploratory model features, not cent-exact reconciliation.
adv_anom_profiles = pd.read_sql_query("""
    SELECT substr(timestamp,1,10) AS day, from_bank, from_account,
           COUNT(*) AS transaction_count, SUM(CAST(amount_paid AS REAL)) AS total_paid_usd,
           AVG(CAST(amount_paid AS REAL)) AS mean_paid_usd,
           MAX(CAST(amount_paid AS REAL)) AS max_paid_usd,
           COUNT(DISTINCT length(to_bank)||':'||to_bank||to_account) AS recipient_count,
           AVG(payment_format='Wire') AS wire_share
    FROM transactions
    WHERE timestamp >= '2022-09-01' AND timestamp < '2022-09-11'
      AND payment_currency='US Dollar' AND receiving_currency='US Dollar'
    GROUP BY substr(timestamp,1,10),from_bank,from_account
    ORDER BY day,from_bank,from_account
""", conn)
adv_anom_keys = ['day','from_bank','from_account']
assert not adv_anom_profiles.duplicated(adv_anom_keys).any()
assert adv_anom_profiles.transaction_count.sum() == adv_anom_population.eligible_rows
assert np.isfinite(adv_anom_profiles.select_dtypes('number').to_numpy()).all()
assert (adv_anom_profiles.select_dtypes('number') >= 0).all().all()
adv_anom_train = adv_anom_profiles.loc[adv_anom_profiles.day < '2022-09-08'].copy()
adv_anom_test = adv_anom_profiles.loc[adv_anom_profiles.day >= '2022-09-08'].copy()
assert adv_anom_train.day.max() < adv_anom_test.day.min()
show_table(pd.DataFrame([
    {'Population': 'Full ledger transactions', 'Rows': int(adv_anom_population.all_rows)},
    {'Population': 'Main-period transactions, all currencies', 'Rows': int(adv_anom_population.main_rows)},
    {'Population': 'Included main-period USD → USD transactions', 'Rows': int(adv_anom_population.eligible_rows)},
    {'Population': 'Excluded main-period USD-paid cross-currency transactions', 'Rows': int(adv_anom_population.excluded_usd_fx_rows)},
    {'Population': 'Earlier fit account-days (September 1–7)', 'Rows': len(adv_anom_train)},
    {'Population': 'Later scored account-days (September 8–10)', 'Rows': len(adv_anom_test)},
]))

In [ ]:
adv_anom_features = ['transaction_count','total_paid_usd','mean_paid_usd',
                     'max_paid_usd','recipient_count','wire_share']
adv_anom_logged_features = adv_anom_features[:-1]
def adv_anom_matrix(frame):
    result = frame[adv_anom_features].copy()
    result[adv_anom_logged_features] = np.log1p(result[adv_anom_logged_features])
    return result
adv_anom_x_train = adv_anom_matrix(adv_anom_train)
adv_anom_x_test = adv_anom_matrix(adv_anom_test)
# The baseline only flags unusually high counts/amounts/diversity, not low activity.
adv_anom_medians = adv_anom_x_train[adv_anom_logged_features].median()
adv_anom_iqrs = (adv_anom_x_train[adv_anom_logged_features].quantile(.75)
                 - adv_anom_x_train[adv_anom_logged_features].quantile(.25))
# A fixed one-log-unit fallback handles features with a zero training IQR.
adv_anom_scales = adv_anom_iqrs.where(adv_anom_iqrs > 1e-9, 1.0)
adv_anom_deviations = ((adv_anom_x_test[adv_anom_logged_features] - adv_anom_medians)
                       / adv_anom_scales).clip(lower=0)
adv_anom_test['baseline_score'] = adv_anom_deviations.max(axis=1)
adv_anom_test['baseline_reason'] = adv_anom_deviations.idxmax(axis=1)
adv_anom_forest = IsolationForest(n_estimators=100,max_samples=256,contamination='auto',
                                 random_state=42,n_jobs=2)
adv_anom_forest.fit(adv_anom_x_train.to_numpy(dtype='float32'))
adv_anom_test['forest_score'] = -adv_anom_forest.score_samples(adv_anom_x_test.to_numpy(dtype='float32'))
adv_anom_k = max(1,int(np.ceil(.01*len(adv_anom_test))))
# Exactly k rows per ranking; deterministic ties use date, bank and account.
for adv_anom_method in ['baseline','forest']:
    adv_anom_order = adv_anom_test.sort_values(
        [adv_anom_method+'_score',*adv_anom_keys],ascending=[False,True,True,True],kind='mergesort').index
    adv_anom_test[adv_anom_method+'_selected'] = False
    adv_anom_test.loc[adv_anom_order[:adv_anom_k],adv_anom_method+'_selected'] = True
adv_anom_tie_rows = []
for adv_anom_method in ['baseline','forest']:
    adv_anom_cutoff = float(adv_anom_test.loc[adv_anom_test[adv_anom_method+'_selected'],adv_anom_method+'_score'].min())
    adv_anom_at_cutoff = adv_anom_test[adv_anom_method+'_score'].eq(adv_anom_cutoff)
    adv_anom_tie_rows.append({'Method':adv_anom_method,'Cutoff score':adv_anom_cutoff,
        'Rows equal to cutoff':int(adv_anom_at_cutoff.sum()),
        'Cutoff rows selected':int((adv_anom_at_cutoff & adv_anom_test[adv_anom_method+'_selected']).sum()),
        'Cutoff rows excluded':int((adv_anom_at_cutoff & ~adv_anom_test[adv_anom_method+'_selected']).sum())})
adv_anom_ties = pd.DataFrame(adv_anom_tie_rows)
show_table(adv_anom_ties)
adv_anom_frozen_scores = adv_anom_test[[*adv_anom_keys,'baseline_score','forest_score',
                                       'baseline_selected','forest_selected']].copy()
adv_anom_seen_accounts = set(zip(adv_anom_train.from_bank,adv_anom_train.from_account))
adv_anom_test['seen_during_fit'] = [key in adv_anom_seen_accounts
                                  for key in zip(adv_anom_test.from_bank,adv_anom_test.from_account)]
show_table(pd.DataFrame({'Feature':adv_anom_features,
    'Transform':['log(1+x)']*5+['Untransformed fraction'],
    'Baseline train median':[adv_anom_medians.get(f,np.nan) for f in adv_anom_features],
    'Baseline train scale':[adv_anom_scales.get(f,np.nan) for f in adv_anom_features]}))
display(Markdown(f"**Frozen ranks:** {len(adv_anom_test):,} later account-days; "
    f"{adv_anom_k:,} selected per method ({adv_anom_k/len(adv_anom_test):.2%}). "
    f"Scikit-learn {adv_anom_version('scikit-learn')}. The forest uses all earlier profiles; "
    "only 256 are sampled inside each tree. This top-1% cutoff uses the complete later batch, "
    "so it represents a retrospective workload budget, not an online threshold. "
    "Rows with exactly equal cutoff scores use date/bank/account order; that arbitrary tie rule "
    "can separate equally scored profiles and does not add evidence of risk."))

### Post-ranking AML diagnostics

Only now join labels. A positive account-day contains **at least one labeled laundering USD → USD outgoing transaction that day**. The denominator below is later account-days, not transactions, candidates or people. “Capture” means the share of positive account-days selected; “positive share” means positive account-days divided by selected account-days; lift compares that share with the later population prevalence. These are descriptive AML-label overlaps, not embezzlement accuracy or validation of a deployed detector. Repeated accounts are dependent, and three test dates provide no reliable generalization estimate.

In [ ]:
adv_anom_labels = pd.read_sql_query("""
    SELECT substr(timestamp,1,10) AS day,from_bank,from_account,
           MAX(is_laundering) AS any_laundering,
           SUM(is_laundering) AS laundering_transactions
    FROM transactions
    WHERE timestamp >= '2022-09-08' AND timestamp < '2022-09-11'
      AND payment_currency='US Dollar' AND receiving_currency='US Dollar'
    GROUP BY substr(timestamp,1,10),from_bank,from_account
""", conn)
adv_anom_test = adv_anom_test.merge(adv_anom_labels,on=adv_anom_keys,how='left',validate='one_to_one')
assert adv_anom_test.any_laundering.notna().all()
adv_anom_prevalence = float(adv_anom_test.any_laundering.mean())
adv_anom_positive_days = int(adv_anom_test.any_laundering.sum())
adv_anom_metrics = []
for adv_anom_method,adv_anom_label in [('baseline','Upper-tail baseline'),('forest','Isolation Forest')]:
    adv_anom_selected = adv_anom_test.loc[adv_anom_test[adv_anom_method+'_selected']]
    adv_anom_hits = int(adv_anom_selected.any_laundering.sum())
    adv_anom_metrics.append({'Method':adv_anom_label,'Selected account-days':len(adv_anom_selected),
        'Positive account-days':adv_anom_hits,'Positive share':adv_anom_hits/len(adv_anom_selected),
        'Capture of positive account-days':adv_anom_hits/adv_anom_positive_days if adv_anom_positive_days else np.nan,
        'Lift over prevalence':(adv_anom_hits/len(adv_anom_selected))/adv_anom_prevalence if adv_anom_prevalence else np.nan,
        'Selected unseen account-days':int((~adv_anom_selected.seen_during_fit).sum()),
        'Selected unseen share':float((~adv_anom_selected.seen_during_fit).mean())})
adv_anom_metrics = pd.DataFrame(adv_anom_metrics)
show_table(adv_anom_metrics,formats={'Positive share':'{:.2%}',
    'Capture of positive account-days':'{:.2%}','Lift over prevalence':'{:.2f}×','Selected unseen share':'{:.2%}'})
adv_anom_overlap = int((adv_anom_test.baseline_selected & adv_anom_test.forest_selected).sum())
display(Markdown(f"Later population: **{adv_anom_positive_days:,} / {len(adv_anom_test):,} "
    f"positive account-days ({adv_anom_prevalence:.3%})**. The two review lists share "
    f"{adv_anom_overlap:,} account-days. Label 0 means no labeled laundering in this scoped daily "
    "outgoing profile; it does not prove that the account or owner is innocent of every offense. "
    f"Overall, {(~adv_anom_test.seen_during_fit).sum():,} / {len(adv_anom_test):,} "
    f"later profiles ({(~adv_anom_test.seen_during_fit).mean():.2%}) belong to accounts "
    "not observed during fitting in this USD scope."))

In [ ]:
adv_anom_fig = make_subplots(rows=1,cols=2,horizontal_spacing=.15,
    subplot_titles=('Amount alone does not define the ranking',
                    'A pooled budget produces different daily rates'))
# Only the displayed reference is reduced; scores, ranks, and all selections stay unchanged.
adv_anom_plot = adv_anom_test.sample(n=min(5000,len(adv_anom_test)),random_state=42)
adv_anom_top = adv_anom_test.loc[adv_anom_test.forest_selected]
for adv_anom_frame,adv_anom_name,adv_anom_color,adv_anom_size in [
    (adv_anom_plot,f'Reference ({len(adv_anom_plot):,} sampled account-days)',REST,4),
    (adv_anom_top,f'All top 1% ({len(adv_anom_top):,} account-days)',FOCUS,5)]:
    adv_anom_fig.add_trace(go.Scatter(
        x=np.log10(1+adv_anom_frame.total_paid_usd),y=adv_anom_frame.forest_score,
        mode='markers',name=adv_anom_name,
        marker=dict(symbol='square',size=adv_anom_size,color=adv_anom_color,line=dict(width=0)),
        customdata=adv_anom_frame[['day','from_bank','from_account','total_paid_usd']].to_numpy(),
        hovertemplate=('Date: %{customdata[0]}<br>Bank / account: %{customdata[1]} / %{customdata[2]}'
            '<br>Daily outgoing USD: $%{customdata[3]:,.2f}'
            '<br>Isolation Forest score: %{y:.5f}<extra>%{fullData.name}</extra>')),
        row=1,col=1)
adv_anom_by_day = adv_anom_test.groupby('day').agg(
    account_days=('day','size'),baseline_selected=('baseline_selected','sum'),forest_selected=('forest_selected','sum'))
adv_anom_day_labels = [pd.Timestamp(d).strftime('%d %b') for d in adv_anom_by_day.index]
for adv_anom_method,adv_anom_color,adv_anom_name in [
    ('baseline',GROUPS[0],'Upper-tail baseline'),('forest',GROUPS[1],'Isolation Forest')]:
    adv_anom_fig.add_trace(go.Bar(
        x=adv_anom_day_labels,
        y=100*adv_anom_by_day[adv_anom_method+'_selected']/adv_anom_by_day.account_days,
        name=adv_anom_name,marker=dict(color=adv_anom_color,line=dict(width=0)),
        customdata=np.column_stack([adv_anom_by_day[adv_anom_method+'_selected'],adv_anom_by_day.account_days]),
        hovertemplate=('%{x} 2022<br>Selected account-days: %{customdata[0]:,}'
            '<br>Daily account-days: %{customdata[1]:,}'
            '<br>Selected share: %{y:.2f}%<extra>%{fullData.name}</extra>')),
        row=1,col=2)
adv_anom_fig.update_layout(barmode='group',bargap=.3)
adv_anom_fig.update_xaxes(title_text='log₁₀(1 + daily outgoing USD)',row=1,col=1)
adv_anom_fig.update_yaxes(title_text='Isolation Forest anomaly score',row=1,col=1)
adv_anom_fig.update_xaxes(title_text='Scored date (2022)',row=1,col=2)
adv_anom_fig.update_yaxes(title_text='Selected / daily account-days (%)',rangemode='tozero',row=1,col=2)
finish_figure(adv_anom_fig,'advanced_anomaly_ranks',
    title='Later-period review ranks reflect more than outgoing amounts',
    subtitle=f"USD → USD outgoing profiles · 8–10 September 2022 · {len(adv_anom_test):,} scored account-days",
    note=f'Reference display: {len(adv_anom_plot):,} account-days sampled with seed 42; all {len(adv_anom_top):,} forest selections appear. '
         'Overlapping points are not jittered. The 1% budget is pooled across dates; cutoff ties use arbitrary date/bank/account order. '
         'Ranks exclude labels and do not establish embezzlement.',
    height=600)

In [ ]:
# This explains the transparent baseline; it is NOT an attribution of the forest score.
adv_anom_display_columns = [*adv_anom_keys,'transaction_count','total_paid_usd','recipient_count',
                           'wire_share','forest_score','baseline_reason','any_laundering']
adv_anom_examples = adv_anom_test.sort_values(
    ['forest_score',*adv_anom_keys],ascending=[False,True,True,True],kind='mergesort').head(10)
show_table(adv_anom_examples[adv_anom_display_columns],formats={
    'total_paid_usd':'${:,.2f}','wire_share':'{:.1%}','forest_score':'{:.4f}'})
adv_anom_summary = {'fit_account_days':len(adv_anom_train),'scored_account_days':len(adv_anom_test),
    'eligible_transactions':int(adv_anom_population.eligible_rows),
    'excluded_usd_fx_transactions':int(adv_anom_population.excluded_usd_fx_rows),
    'positive_test_account_days':adv_anom_positive_days,'test_account_day_prevalence':adv_anom_prevalence,
    'selected_per_method':adv_anom_k,'rank_overlap':adv_anom_overlap,
    'runtime_seconds':round(time.perf_counter()-adv_anom_started,2)}
show_table(pd.DataFrame({'Metric':list(adv_anom_summary),
                         'Value':[f'{v:,.0f}' if isinstance(v,int) else f'{v:,.5f}'
                                  for v in adv_anom_summary.values()]}))
# Verify that joining labels did not alter frozen scores or review decisions.
adv_anom_recheck = adv_anom_test[adv_anom_frozen_scores.columns].sort_values(adv_anom_keys).reset_index(drop=True)
pd.testing.assert_frame_equal(adv_anom_frozen_scores.sort_values(adv_anom_keys).reset_index(drop=True),adv_anom_recheck)

**How to investigate these ranks.** Large legitimate payroll, treasury, merchant settlement or investment flows can look unusual. Review the actual transaction list and compare invoices, counterparties, approvals and beneficial ownership before escalating a case. The top rows are review examples, not allegations.

**Limits.** Correlated volume features give large/high-activity profiles substantial influence; wire share and unusual combinations can also affect the forest. Training activity is not certified benign, peer types are mixed, first-seen accounts may rank differently, and the model does not compare an account against its own past. Account-day aggregation loses payment ordering, incoming flows, split payments and graph context. USD → USD excludes other currencies and conversion flows. SQL floating-point totals support ranking only, not exact money tracing. Daily observations can repeat the same account, and the short chronological split does not remove synthetic-generator bias. No external test set, calibrated probabilities, threshold optimization or causal/embezzlement validation is claimed. Fit/score dates are disjoint; the same account may occur in both periods, as in monitoring known customers.

## Does the annotated-pattern subset contain the target shape?

This separate retrospective diagnostic checks two-payment chains **within each labelled laundering attempt**, using the same 72-hour, currency, ratio, distinct-account and distinct-entity conditions. Pattern annotations enter this evaluation only; they never enter candidate selection. This subset cannot measure full-ledger recall because it omits unannotated and cross-attempt chains.

In [ ]:
context_pattern_path = REPORT_DIR / 'hi-small-pattern-scope.json'
if not context_pattern_path.exists():
    display(Markdown('The optional pattern-scope report is unavailable. Within-attempt scope checks were not run in this notebook; no zero-match result is inferred.'))
else:
    context_pattern_bytes = (REPORT_DIR/'hi-small-pattern-scope.json').read_bytes()
    context_pattern_artifact = context_run.get('artifacts', {}).get('hi-small-pattern-scope.json')
    if context_pattern_artifact is not None:
        assert hashlib.sha256(context_pattern_bytes).hexdigest() == context_pattern_artifact['sha256']
    context_patterns = json.loads(context_pattern_bytes)
    for context_source in context_patterns['sources'].values():
        assert context_source['sha256'] == context_acquisition_by_name[context_source['file_name']]['sha256']
    context_pattern_table = pd.DataFrame([
        {'Pattern type':name, 'Attempts':v['attempt_count'], 'Annotated rows':v['pattern_row_count'],
         'Eligible pairs':v['eligible_pair_count'], 'Matched attempts':v['matched_attempt_count']}
        for name,v in context_patterns['by_pattern_type'].items()])
    assert int(context_pattern_table['Attempts'].sum()) == AUDIT['patterns']['attempts']
    assert int(context_pattern_table['Annotated rows'].sum()) == AUDIT['patterns']['transaction_rows']
    show_table(context_pattern_table, formats={n:'{:,.0f}' for n in
                                             ['Attempts','Annotated rows','Eligible pairs','Matched attempts']})
    context_pattern_rows = context_patterns['scope']['pattern_row_count']
    context_pattern_persons = context_patterns['scope']['pattern_account_roles'].get('person',0)
    display(Markdown(f"The {context_patterns['scope']['attempt_count']:,} attempts contain "
                     f"{context_pattern_rows:,} annotated rows and **{context_pattern_persons:,} person-classified accounts**. "
                     f"Of {context_patterns['funnel']['connected_time_window_pairs']:,} connected pairs within the time window, "
                     f"{context_patterns['eligible_pair_count']:,} meet the full role-and-payment rule. "
                     "The business → business → person hypothesis does not occur in this annotated subset. "
                     "Zero matches is a scope mismatch, not proof that the scanner missed every laundering pattern.\n\n"
                     f"The pattern-row count ({context_pattern_rows:,}) is smaller than the ledger's "
                     f"{int(context_db_counts['label_1']):,} labelled-laundering rows: a count gap of "
                     f"{int(context_db_counts['label_1'])-context_pattern_rows:,}. "
                     "The audit found every published pattern signature in the ledger, but this diagnostic "
                     "does not establish one-to-one attribution for every labelled ledger row."))

## What the data can support

The selected paths describe a payment shape. Neither a path nor a simulator label establishes embezzlement, a shell vendor, employee involvement, a false invoice, or an unauthorized payment. Thresholds were fixed before evaluating labels. Label overlap is descriptive; it is not validated accuracy for the user's embezzlement scenario.

| ACFE-informed check | Evidence needed | HI-Small status |
|---|---|---|
| Employee/vendor links | Employee/vendor records and verified account or ownership links | **Not run — source data unavailable** |
| Invoice / purchase-order / receipt consistency | Invoices, orders and delivery or acceptance evidence | **Not run — source data unavailable** |
| Purchase/payment approval separation | Approver identities, authority and exceptions | **Not run — source data unavailable** |
| Shell-company or beneficial-owner assessment | Filings, operational evidence and corroborated ownership | **Not established by synthetic names** |
| Embezzlement case evaluation | Independently reviewed case outcomes | **Not computable — no embezzlement case labels** |

**Interpretation limits:** Roles come from name prefixes and lack effective dates. Missing/unknown roles narrow the observable hypothesis. Minute-resolution timestamps exclude equal-time hops. The 72-hour/0.80–1.05 filters are engineering choices, not ACFE thresholds. The adjacent intermediary currencies must match, but either transfer may itself convert currencies. This two-hop rule does not attribute value across mismatched intermediary currencies or reconstruct longer paths and split/merged flows. Similar amounts do not prove that the same money moved. Overlapping paths are not independent observations: their count is not a case count, and their amounts must not be summed into losses. Consider legitimate explanations and evidence that weakens an allegation.

**Sources:** [IBM dataset on Kaggle](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml), [IBM AML-Data repository](https://github.com/IBM/AML-Data), [IBM dataset paper](https://arxiv.org/html/2306.16424v3), [ACFE Anti-Fraud Data Analytics Tests](https://www.acfe.com/fraud-resources/fraud-risk-tools---coso/anti-fraud-data-analytics-tests), [ACFE Shell Companies](https://www.acfe.com/fraud-resources/shell-companies), and [CFE professional standards and interpretation](https://www.acfe.com/about-the-acfe/-/media/9560BECA7A1441B5B3CD9A6E14CE7B89.ashx). Retrieval records and hashes remain with the local analysis.

## Reproducibility and next experiments

The headline counts use the complete ledger. The amount-distribution reference is deliberately smaller and cannot estimate prevalence. Candidate concentration describes overlapping paths, not independent fraud incidents or calculated losses.

Useful next experiments require independently labeled company/employee relationships, authorized invoices and approvals, and legitimate supplier-payroll lookalikes. Compare predefined alternatives using chronological evaluation and report missing coverage. Keep AML labels and pattern membership outside feature construction and candidate selection.

This notebook saves its figures under `reports/eda/figures/`. The source notebook keeps outputs cleared in Git; the executed copy and HTML preview stay under `reports/eda/` with the other local analysis outputs.

In [ ]:
assert len(FIGURES) == 10
(REPORT_DIR / 'eda/figure-manifest.json').write_text(json.dumps(FIGURES, indent=2))
assert SCAN['full_coverage'] is True and SCAN['truncated'] is False
assert len(pairs) == SCAN['candidate_count']
assert len(selected_ids) == SCAN['selected_edge_diagnostics']['total']
conn.rollback()
conn.close()
display(Markdown('EDA completed. All source reconciliation and saved-candidate checks passed; the database connection is closed.'))